# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The notebook follows the Croissant standard for referencing record sets, fields, and columns by their unique `@id`.

### Dataset Source
The dataset is described by a Croissant schema and available here:  
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure mlcroissant is installed. You may want to restart the kernel after installation.!pip install mlcroissant

## 1. Data Loading
We start by importing the required libraries and loading the dataset's metadata with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:\n", metadata.description)
print("\nVersion:", metadata.version)
print("Date published:", metadata.datePublished)
print("Identifier:", metadata.identifier)


## 2. Data Overview
Now let's review the available record sets and their fields by their unique `@id`s to understand what data is available.

*Note: All references to dataset entities (record sets, fields, columns) will use their `@id` as per the Croissant specification.*

In [ ]:
# List all record sets in the dataset by their @id and name

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    if isinstance(metadata.recordSet, dict):
        record_sets = [metadata.recordSet]
    else:
        record_sets = metadata.recordSet
    print("Record sets (@id, name):")
    for rec in record_sets:
        print(f"  - {rec['@id']}: {rec.get('name', 'No name')}")
else:
    # If the Croissant package doesn't include recordSet in the metadata, try listing via the dataset API
    from mlcroissant.structures.croissant import get_record_sets_from_metadata
    record_sets = get_record_sets_from_metadata(metadata)
    print("Record sets (@id, name):")
    for rec in record_sets:
        print(f"  - {rec['@id']}: {rec.get('name', 'No name')}")

# For this dataset, as of 2024-06, try detecting available record sets directly
# Use mlcroissant's API to inspect available record sets if not found in metadata

print("\nAvailable record set IDs for use:")
available_record_sets = dataset.record_sets()
for rset in available_record_sets:
    print('-', rset)

# For each record set, show a preview of records with their @id
for record_set_id in available_record_sets:
    print(f"\nFields for record set @id: {record_set_id}")
    example = next(dataset.records(record_set=record_set_id), None)
    if example:
        print('Fields:', list(example.keys()))
        print('Sample:', example)
    else:
        print('No records found in this record set.')

## 3. Data Extraction
Let's load the data from all main record sets into Pandas DataFrames, using their unique record set `@id`. We'll preview available fields for analysis. You can later select specific record sets and fields of interest using these IDs.

In [ ]:
# Retrieve all available record set @id's from the dataset
record_set_ids = dataset.record_sets()
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

print(f"\nLoaded DataFrames for record sets (@id):", list(dataframes.keys()))

# For demonstration, view columns and a preview for the first record set
if record_set_ids:
    example_rsid = record_set_ids[0]
    print(f"\nColumns for record set {example_rsid}:")
    print(dataframes[example_rsid].columns.tolist())
    display(dataframes[example_rsid].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
We now apply common preprocessing and EDA operations. This includes filtering numeric fields, normalizing values, and optionally grouping by a categorical field. 

Below, replace `<numeric_field_id>` and `<group_field_id>` with the target field (column) `@id` as discovered above for your actual analysis.

In [ ]:
# Choose the specific record set you want to analyze
# Replace with your own if you wish to analyze a different one
record_set_id = record_set_ids[0] if record_set_ids else None

df = dataframes[record_set_id].copy()
print(f"Exploring record set: {record_set_id} ({len(df)} rows)")

# Choose a numeric field and a grouping field by their @id
# For demonstration, detect numeric columns
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
print("Available numeric fields:", numeric_fields)

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use first as example
    threshold = df[numeric_field_id].mean()  # e.g., use mean as a threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()

    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Find possible grouping fields (categorical)
    possible_group_fields = df.select_dtypes(exclude=['number']).columns.tolist()
    print("Available grouping fields:", possible_group_fields)

    if possible_group_fields:
        group_field = possible_group_fields[0]  # as example
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields available for EDA in this record set.")

## 5. Visualization
Let's visualize the distribution of a selected numeric field and plot the mean per group (if a suitable grouping field is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Distribution plot of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is present, plot mean per group
    if possible_group_fields:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library. We:
* Loaded dataset metadata and retrieved available record sets and their fields via their `@id`.
* Extracted record data as DataFrames for further analysis.
* Performed basic exploratory analysis including numeric filtering, normalization, grouping, and visualization of dataset variables using only their Croissant `@id`.

For custom analyses, adjust the record set, field, and grouping `@id` based on the schema, and refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) for further usage.